# EGC KEGG Tree

Builds the KEGG annotation tree from `kegg_db.json` and writes one `.ko.json`
per sample to `DataStructures/Kegg/`.

**Run order:** top to bottom, selecting the KEGG INPUT cell (not the Taxonomy one).


In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r'C:\Users\sakoehn\Desktop\JGI\Notebooks')
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc


## 1. Load sample list and kegg_db


In [ ]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f'Loaded {len(samples)} samples')

kegg_db_path    = Path(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json')
metagenome_base = Path(r'C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples')

# E6 (docs/AUDIT.md): kegg_db.json is a prebuilt intermediate (produced by
# Build_KEGG_DB.ipynb from the raw .a.ko.txt files), unlike the other three
# EGC notebooks, which read their raw files directly every run. A stale
# intermediate would silently give stale KEGG EGC with no visible error --
# fail loudly here instead if any real input is newer than it.
kegg_db_mtime = kegg_db_path.stat().st_mtime
stale_inputs = [
    s for s in samples
    if (metagenome_base / s / f'{s}.a.ko.txt').exists()
    and (metagenome_base / s / f'{s}.a.ko.txt').stat().st_mtime > kegg_db_mtime
]
if stale_inputs:
    raise RuntimeError(
        f"kegg_db.json is older than {len(stale_inputs)} of its real .a.ko.txt "
        f"inputs (e.g. {stale_inputs[0]}) -- re-run Build_KEGG_DB.ipynb before this notebook."
    )

with open(kegg_db_path) as f:
    kegg_db = json.load(f)
print(f'kegg_db: {len(kegg_db):,} genes')


## 1.2. Load KEGG BRITE hierarchy (ko00001.keg)

Parses the hierarchy file into `brite` (one entry per node -- name, depth,
category) and `brite_occurances` (every occurrence of a node, each with the
parent it had *at that occurrence*). A KO can legitimately appear under
several different parents in this file, so `brite_occurances` is where that
gets tracked -- this is the actual source of truth the tree-build step below
walks; it doesn't do any tracking of its own.


In [3]:
with open(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\ko00001.keg') as f:
    keg_lines = f.readlines()

brite = {}          # {node_id: {name, depth, category, parent_id, parent_name}}
brite_occurances = {} # {node_id : [ {parent_id, parent_name, top_level, branch_id, branch_name}, ...]}
current_at_depth = {}

for line in keg_lines:
    if not any(line.startswith(c) for c in 'ABCDEFGHIJK'):
        continue
    level_char = line[0]
    depth      = ord(level_char) - ord('A') + 1
    rest       = line[1:].strip()
    if not rest:
        continue
    parts = rest.split(None, 1)
    if len(parts) < 2:
        continue
    node_id, name = parts[0], parts[1]
    parent        = current_at_depth.get(depth - 1)
    category      = name if depth == 1 else current_at_depth.get(1, {}).get('name')
    branch        = current_at_depth.get(2)  # B-level ancestor for this occurrence, e.g. 09159 Environmental adaptation
    current_at_depth[depth] = {'id': node_id, 'name': name}
    for d in list(current_at_depth):
        if d > depth:
            del current_at_depth[d]
    brite[node_id] = {
        'name':        name,
        'depth':       depth,
        'category':    category,
        'parent_id':   parent['id']   if parent else None,
        'parent_name': parent['name'] if parent else None,
    }
    if node_id not in brite_occurances:
        brite_occurances[node_id] = []
    brite_occurances[node_id].append({'parent_id':   parent['id']   if parent else None,
                                       'parent_name': parent['name'] if parent else None,
                                       'top_level':   category,
                                       'branch_id':   branch['id']   if branch else None,
                                       'branch_name': branch['name'] if branch else None})

print(f'BRITE: {len(brite):,} nodes')


BRITE: 28,893 nodes


## 2. Build KEGG tree

A KO can appear under more than one parent in `ko00001.keg` (an enzyme that's
a member of several pathways, say) -- `brite_occurances[node_id]` (built
above) holds every occurrence, not just one.

For each gene, this walks from its KO upward through *every* occurrence's
parent, using a to-do list of nodes still to visit rather than a single
"current" pointer, since a KO with N parents needs to explore N directions
at once. A `visited` set, scoped to one gene's walk, stops the same ancestor
being re-queued when two different parents converge on it (e.g. both
Glycolysis and Galactose metabolism roll up into Carbohydrate metabolism) --
registration only happens the first time a node is actually visited, so a
gene's coverage is counted at most once per node no matter how many paths
reach it, while still counting once at every distinct node it's really a
member of.


In [ ]:
tax_tree = {}  # {node_id: {name, category, depth, parent_id, parent_name, timepoints}}
dropped_ko_ids = set()  # E9 (docs/AUDIT.md): real KO ids absent from ko00001.keg's brite dict

start = time.time()

for gene_id, ginfo in kegg_db.items():
    scaffold   = gene_id.rsplit('_', 2)[0]
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])

    for full_path, sample in zip(lookups, timepoints):
        # E10 (docs/AUDIT.md): assert exactly one id per entry instead of
        # silently taking only the first of a ';'-joined list -- checked
        # against real data, this never currently fires (every entry has
        # exactly one id), so if it ever does, that's a real format change
        # worth failing loudly on rather than silently dropping ids.
        segments = full_path.split(';')
        assert len(segments) == 1, (
            f"Expected exactly one KEGG id per lookupId entry, got "
            f"{len(segments)} in {full_path!r} (gene {gene_id}) -- "
            f"the raw .a.ko.txt format may have changed."
        )
        ko_id = segments[0].strip()
        if ko_id.startswith('KO:'):
            ko_id = ko_id[3:]

        if ko_id not in brite:
            # E9 (docs/AUDIT.md): a truncated ko00001.keg download is
            # indistinguishable from a genuinely obsolete id -- can't tell
            # which without a real count. Skip is still the right behavior
            # (can't classify what isn't in the reference), just make it
            # visible instead of silent.
            dropped_ko_ids.add(ko_id)
            continue

        to_visit = [ko_id]
        visited  = set()

        while to_visit:
            current = to_visit.pop()
            if current in visited:
                continue
            visited.add(current)
            if current not in brite:
                continue

            if current not in tax_tree:
                b = brite[current]
                tax_tree[current] = {
                    'name':        b['name'],
                    'category':    b['category'],
                    'depth':       b['depth'],
                    'parent_id':   b['parent_id'],
                    'parent_name': b['parent_name'],
                    'timepoints':  {},
                }

            node = tax_tree[current]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Only count this gene once per (KO node, sample) — a gene can
            # appear multiple times if it has multiple KO hits in kegg_db.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaffold] = tp['scaffolds'].get(scaffold, 0) + 1
            tp['genes'].add(gene_id)

            for occ in brite_occurances[current]:
                parent_id = occ['parent_id']
                if parent_id is None:
                    continue  # this occurrence IS a root, nothing above it
                if parent_id not in visited:
                    to_visit.append(parent_id)

print(f'KEGG tree: {len(tax_tree):,} nodes  ({time.time() - start:.1f}s)')
print(f"Dropped {len(dropped_ko_ids)} distinct KO ids not found in ko00001.keg (see docs/MISSING_ANNOTATIONS.md)")
if dropped_ko_ids:
    print(f"  e.g. {sorted(dropped_ko_ids)[:10]}")

## 3. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a KEGG term (from this notebook's own `kegg_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_COG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a KEGG hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a KEGG-only subset of the shared total) can never exceed it.

In [5]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_COG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [6]:
def avg_gene_map(node_id):
    node = tax_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in tax_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 4. Write .ko.json files


In [7]:
base = Path(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg')
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for node_id, node in tax_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_id] = {
            'name':        node['name'],
            'category':    node['category'],
            'depth':       node['depth'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp['egc'],
            'norm_egc':    tp['norm_egc'],
            'scaffolds':   tp.get('scaffolds', {}),
        }
    out_path = base / f'{s}.ko.json'
    with open(out_path, 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f'Wrote {len(samples)} .ko.json files to {base}')


Wrote 128 .ko.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg
